# Compliance Breach Detection Analysis

This notebook provides exploratory analysis for the compliance breach detection workflow.

## Overview

In this notebook, we explore:
1. The synthetic dataset of compliance messages
2. Model performance on different categories
3. Error analysis and common patterns
4. Prompt optimization opportunities

In [ ]:
# Import necessary libraries
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import google.cloud.aiplatform as vertexai
import mlflow

# Configure plot style
plt.style.use('ggplot')
sns.set(style="whitegrid")

# Load utility functions from the project
import sys
sys.path.append('/home/seanm/repos/llm_ops_pipeline')
from llm_ops_pipeline.utils.logging import setup_logging
from llm_ops_pipeline.config.config import load_config
from workflows.compliance_detection.scripts.compliance_predictor import CompliancePredictor

In [ ]:
# Configure paths and load data
CONFIG_PATH = "../configs/compliance_config.yaml"
DATA_DIR = "../../../data/compliance"
EVAL_DIR = "../evaluation"

# Load configuration
config = load_config(CONFIG_PATH)
logger = setup_logging(name="notebook", level="INFO")

# Load data
def load_data(split):
    with open(f"{DATA_DIR}/{split}/data.json", 'r') as f:
        return json.load(f)

train_data = load_data("train")
val_data = load_data("validation")
test_data = load_data("test")

# Create DataFrames
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)
test_df = pd.DataFrame(test_data)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

## Dataset Exploration

Let's explore the characteristics of our synthetic dataset.

In [ ]:
# Distribution of categories
plt.figure(figsize=(12, 6))
train_df['label'].value_counts().plot(kind='bar')
plt.title('Distribution of Categories in Training Data')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Text length distribution
train_df['text_length'] = train_df['text'].apply(len)
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(12, 6))
sns.histplot(data=train_df, x='word_count', hue='label', multiple='stack', bins=20)
plt.title('Distribution of Word Count by Category')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Summary statistics
print("Text length statistics:")
print(train_df.groupby('label')['text_length'].describe())
print("\nWord count statistics:")
print(train_df.groupby('label')['word_count'].describe())

## Sample Messages by Category

Let's look at some examples from each category.

In [ ]:
for category in config["data"]["categories"]:
    print(f"\n=== {category} ===\n")
    samples = train_df[train_df['label'] == category].sample(3)['text'].values
    for i, sample in enumerate(samples):
        print(f"{i+1}. {sample}\n")

## Common Words Analysis by Category

Let's analyze the most common words in each category to understand distinguishing features.

In [ ]:
# Download NLTK resources if needed
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def get_top_words(texts, n=20):
    words = []
    for text in texts:
        tokens = word_tokenize(text.lower())
        words.extend([word for word in tokens if word.isalpha() and word not in stop_words])
    
    word_freq = pd.Series(words).value_counts().head(n)
    return word_freq

plt.figure(figsize=(15, 20))
for i, category in enumerate(config["data"]["categories"]):
    category_texts = train_df[train_df['label'] == category]['text'].values
    top_words = get_top_words(category_texts)
    
    plt.subplot(3, 2, i+1)
    top_words.plot(kind='barh')
    plt.title(f'Top Words for {category}')
    plt.xlabel('Frequency')
    
plt.tight_layout()
plt.show()

## Model Evaluation Results

Let's analyze the model's performance on the evaluation dataset.

In [ ]:
# Load evaluation results
try:
    with open(f"{EVAL_DIR}/evaluation_results.json", 'r') as f:
        eval_results = json.load(f)
    
    # Display overall metrics
    print("=== Overall Metrics ===")
    print(f"Accuracy: {eval_results['accuracy']:.4f}")
    print(f"Precision (macro): {eval_results['precision_macro']:.4f}")
    print(f"Recall (macro): {eval_results['recall_macro']:.4f}")
    print(f"F1 Score (macro): {eval_results['f1_macro']:.4f}")
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    cm = np.array(eval_results['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=config["data"]["categories"], 
                yticklabels=config["data"]["categories"])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    # Plot category metrics
    metrics_data = {
        "Precision": [eval_results["precision_by_category"][cat] for cat in config["data"]["categories"]],
        "Recall": [eval_results["recall_by_category"][cat] for cat in config["data"]["categories"]],
        "F1": [eval_results["f1_by_category"][cat] for cat in config["data"]["categories"]]
    }
    
    df = pd.DataFrame(metrics_data, index=config["data"]["categories"])
    df.plot(kind='bar', figsize=(12, 6))
    plt.title('Metrics by Category')
    plt.ylabel('Score')
    plt.xlabel('Category')
    plt.ylim(0, 1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(loc='best')
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("Evaluation results not found. Run the evaluation script first.")

## Error Analysis

Let's analyze the model's errors to understand where it struggles.

In [ ]:
try:
    # Extract error cases
    errors_df = pd.DataFrame(eval_results['errors'])
    
    # Error distribution by category
    plt.figure(figsize=(10, 6))
    errors_df['true_label'].value_counts().plot(kind='bar')
    plt.title('Error Distribution by True Category')
    plt.xlabel('Category')
    plt.ylabel('Error Count')
    plt.tight_layout()
    plt.show()
    
    # Most common misclassifications
    misclass_counts = errors_df.groupby(['true_label', 'predicted_label']).size().reset_index(name='count')
    misclass_counts = misclass_counts.sort_values('count', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(data=misclass_counts.head(10), x='count', y='true_label', hue='predicted_label')
    plt.title('Top 10 Misclassification Patterns')
    plt.xlabel('Count')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.show()
    
    # Display some error examples
    print("\n=== Sample Error Cases ===\n")
    for i, row in errors_df.sample(5).iterrows():
        print(f"Text: {row['text']}")
        print(f"True: {row['true_label']}")
        print(f"Predicted: {row['predicted_label']}")
        print("-" * 80)
    
except (FileNotFoundError, KeyError):
    print("Error analysis data not found. Run the evaluation script first.")

## Live Model Testing

Let's test the model on some custom examples.

In [ ]:
# Initialize predictor
predictor = CompliancePredictor(CONFIG_PATH)

# Test examples
test_examples = [
    "Let's meet tomorrow to discuss the Q1 results that we'll be publishing next week.",
    "I can guarantee this investment will yield at least 30% returns, no questions asked.",
    "Make sure to move those funds through our Cayman Islands accounts to avoid detection.",
    "I'm sending you all our client data including SSNs and account balances for your marketing campaign.",
    "That female intern is only here because she looks good, not because of her skills.",
    "We need to inform the regulators before we announce the merger to the public."
]

# Get predictions
for example in test_examples:
    result = predictor.predict(example)
    print(f"Text: {example}")
    print(f"Prediction: {result['prediction']}")
    print(f"Latency: {result['latency_ms']:.2f} ms")
    print("-" * 80)

## MLflow Experiment Tracking

Let's look at our MLflow experiments for prompt tuning.

In [ ]:
# Connect to MLflow
mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])
mlflow.set_experiment(config["mlflow"]["experiment_name"])

# Get experiment ID
experiment = mlflow.get_experiment_by_name(config["mlflow"]["experiment_name"])
if experiment:
    experiment_id = experiment.experiment_id
    
    # Get all runs
    runs = mlflow.search_runs(experiment_ids=[experiment_id])
    
    # Display results
    if not runs.empty:
        # Filter relevant columns
        relevant_cols = ['run_id', 'start_time', 'metrics.accuracy', 'metrics.f1_macro', 
                        'params.prompt_name', 'tags.mlflow.runName']
        filtered_runs = runs[relevant_cols]
        
        # Sort by f1_macro
        filtered_runs = filtered_runs.sort_values('metrics.f1_macro', ascending=False)
        
        # Display table
        print("Top performing prompts:")
        display(filtered_runs.head(10))
        
        # Plot performance comparison
        plt.figure(figsize=(10, 6))
        sns.barplot(data=filtered_runs, x='params.prompt_name', y='metrics.f1_macro')
        plt.title('F1 Score by Prompt Template')
        plt.xlabel('Prompt Template')
        plt.ylabel('F1 Score (macro)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("No runs found in the experiment")
else:
    print("Experiment not found. Run prompt tuning script first.")

## Conclusion and Next Steps

Based on our analysis, we can see areas where the model performs well and areas for improvement. Here are some next steps:

1. **Data Quality**: Generate more examples for categories with lower performance
2. **Prompt Engineering**: Refine prompts for commonly confused categories
3. **Human Review**: Implement human review workflow for low confidence predictions
4. **Model Selection**: Test with other Vertex AI models (e.g., Gemini Pro) to compare performance
5. **Monitoring**: Set up drift detection for the production model